# North Carolina Energy Analysis
## Project Overview

This project analyzes electricity demand in North Carolina and explores how weather patterns influence energy usage. Using data from the U.S. Energy Information Administration (EIA) and Open-Meteo, the project collects hourly and daily electricity demand along with historical weather data. The data is cleaned, merged, and analyzed to identify relationships between temperature, weather conditions, and electricity demand. The long-term goal is to build interactive visualizations and dashboards that provide insights into energy consumption trends across North Carolina.

In [3]:
import pandas as pd
import requests as rq
import json

import datetime as dt
import prettytable
import sqlite3 as sql3
import plotly.express as px
from dotenv import load_dotenv
import os
import openmeteo_requests
import requests_cache
from retry_requests import retry
import numpy as np

load_dotenv()

EIA_API_KEY = os.getenv("EIA_API_KEY")

## Data Collection

In [5]:
prettytable.DEFAULT = 'DEFAULT'
conn = sql3.connect("North_Carolina_Energy_DB.db")
cursor = conn.cursor()
%load_ext sql
%sql sqlite:///North_Carolina_Energy_DB.db

In [6]:
today = dt.datetime.now()
from_date = today - dt.timedelta(days=30)

In [7]:
eia_start = from_date.strftime("%Y-%m-%d")
eia_end = dt.datetime.now().strftime("%Y-%m-%d")
eia_api_length = 720
eia_hourly_start = ""
eia_hourly_end = ""
eia_hourly_length = 720

### Energy Demand

#### Daily Demand

In [10]:
eia_daily_url = f"https://api.eia.gov/v2/electricity/rto/daily-region-data/data/?frequency=daily&data[0]=value&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length={eia_api_length}&api_key={EIA_API_KEY}"
eia_daily_response = rq.get(eia_daily_url)
daily_data = eia_daily_response.json()
eia_daily_data = daily_data['response']['data']

#### Hourly Demand

In [12]:
eia_hourly_url = f"https://api.eia.gov/v2/electricity/rto/region-data/data/?frequency=hourly&data[0]=value&facets[type][]=D&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length={eia_hourly_length}&api_key={EIA_API_KEY}"
eia_hourly_response = rq.get(eia_hourly_url)
hourly_data = eia_hourly_response.json()
eia_hourly_data = hourly_data['response']['data']

### Weather

#### Daily Weather

In [15]:

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
daily_weather_url = "https://archive-api.open-meteo.com/v1/archive"
daily_params = {
	"latitude": [35.5951, 36.0999, 35.2271, 35.7796, 34.2104, 34.8526, 34.0007, 34.1954, 32.7765, 33.6891],
	"longitude": [-82.5515, -80.2442, -80.8431, -78.6382, -77.8868, -82.3940, -81.0348, -79.7626, -79.9311, -78.8867],
	"start_date": eia_start,
	"end_date": eia_end,
    "daily": ["temperature_2m_max", "temperature_2m_min", "wind_speed_10m_max", "wind_speed_10m_min", "precipitation_sum"],
    "temperature_unit": "fahrenheit",
    "wind_speed_unit": "mph",
    "precipitation_unit": "inch"
}
daily_weather_responses = openmeteo.weather_api(daily_weather_url, params = daily_params)

#### Hourly Weather

In [17]:
# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
hourly_weather_url = "https://archive-api.open-meteo.com/v1/archive"
hourly_params = {
	"latitude": 35.7796,
	"longitude": -78.6382,
	"start_date": eia_start,
	"end_date": eia_end,
    "hourly": ["temperature_2m", "relative_humidity_2m", "wind_speed_10m", "cloud_cover", "precipitation"],
    "temperature_unit": "fahrenheit",
    "wind_speed_unit": "mph",
    "precipitation_unit": "inch"
}
hourly_weather_responses = openmeteo.weather_api(hourly_weather_url, params = hourly_params)

### Energy Generation

#### Daily Energy Generation

In [20]:
eia_daily_generation_url = f"https://api.eia.gov/v2/electricity/rto/daily-fuel-type-data/data/?frequency=daily&data[0]=value&facets[fueltype][]=COL&facets[fueltype][]=NG&facets[fueltype][]=NUC&facets[fueltype][]=OIL&facets[fueltype][]=OTH&facets[fueltype][]=SUN&facets[fueltype][]=WAT&facets[fueltype][]=WND&facets[respondent][]=CAR&facets[timezone][]=Eastern&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={EIA_API_KEY}"
eia_daily_generation_response = rq.get(eia_daily_generation_url)
daily_generation_data = eia_daily_generation_response.json()
eia_daily_generation_data = daily_generation_data['response']['data']

#### Hourly Energy Generation

In [22]:
eia_hourly_generation_url = f"https://api.eia.gov/v2/electricity/rto/fuel-type-data/data/?frequency=hourly&data[0]=value&facets[fueltype][]=COL&facets[fueltype][]=NG&facets[fueltype][]=NUC&facets[fueltype][]=OIL&facets[fueltype][]=OTH&facets[fueltype][]=SUN&facets[fueltype][]=WAT&facets[fueltype][]=WND&facets[respondent][]=CAR&start={eia_start}&end={eia_end}&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={EIA_API_KEY}"
eia_hourly_generation_response = rq.get(eia_hourly_generation_url)
hourly_generation_data = eia_hourly_generation_response.json()
eia_hourly_generation_data = hourly_generation_data['response']['data']

## Data Cleaning

### Energy Demand

#### Daily Energy Demand

In [26]:
d_rows = []
for item in eia_daily_data:
    if item['type'] == "D" and item['timezone'] == "Eastern":
        d_rows.append({
            "Date": pd.to_datetime(item["period"]),
            "Demand (MWh)": item["value"]
    })
eia_daily_df = pd.DataFrame(d_rows)
eia_daily_df.dropna(inplace=True)

#### Hourly Demand

In [28]:
h_rows = []
for item in eia_hourly_data:
    h_rows.append({
        "Datetime": pd.to_datetime(item["period"]),
        "Date": pd.to_datetime(item["period"]).date(),
        "Hour": pd.to_datetime(item["period"]).hour,
        "Demand (MWh)": item["value"]
    })
eia_hourly_df = pd.DataFrame(h_rows)
eia_hourly_df.dropna(inplace=True)

### Weather

#### Daily Weather

In [31]:
# Process daily data. The order of variables needs to be the same as requested.
high_temp = []
low_temp = []
mx_w_s = []
mn_w_s = []
prcp_sum = []

for response in daily_weather_responses:
    daily = daily_weather_response.Daily()
    daily_temperature_2m_max = daily.Variables(0).ValuesAsNumpy()
    daily_temperature_2m_min = daily.Variables(1).ValuesAsNumpy()
    daily_wind_speed_10m_max = daily.Variables(2).ValuesAsNumpy()
    daily_wind_speed_10m_min = daily.Variables(3).ValuesAsNumpy()
    daily_precipitation_sum = daily.Variables(4).ValuesAsNumpy()
    high_temp.append(daily_temperature_2m_max)
    low_temp.append(daily_temperature_2m_min)
    mx_w_s.append(daily_wind_speed_10m_max)
    mn_w_s.append(daily_wind_speed_10m_min)
    prcp_sum.append(daily_precipitation_sum)

daily_data = {
	"date": pd.date_range(
		start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = daily.Interval()),
		inclusive = "left"
	).date,
    "high temp": np.mean(high_temp, axis=0).astype(int),
    "low temp": np.mean(low_temp, axis=0).astype(int),
    "max wind speed (mph)": np.round(np.mean(mx_w_s, axis=0).astype(float),2),
    "min wind speed (mph)": np.round(np.mean(mn_w_s, axis=0).astype(float),2),  
    "precipitation sum (inches)": np.round(np.mean(prcp_sum).astype(float),2)
}

daily_weather_df = pd.DataFrame(data = daily_data)

NameError: name 'daily_weather_response' is not defined

#### Hourly Weather

In [ ]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly_temp = []
hourly_hum = []
hourly_w_s = []
hourly_cl_c = []
hourly_precip = []

for response in hourly_weather_responses:
    hourly = hourly_weather_response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(2).ValuesAsNumpy()
    hourly_cloud_cover = hourly.Variables(3).ValuesAsNumpy()
    hourly_precipitation = hourly.Variables(4).ValuesAsNumpy()
    hourly_temp.append(hourly_temperature_2m)
    hourly_hum.append(hourly_humidity_2m)
    hourly_w_s.append(hourly_wind_speed_10m)
    hourly_cl_c.append(hourly_cloud_cover)
    hourly_precip.append(hourly_precipitation)


hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).date,
    "hour": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	).hour,
    "temperature": np.mean(hourly_temp, axis=0).astype(int),
    "humidity %": np.round(np.mean(hourly_hum, axis=0).astype(float),2),
    "wind speed (mph)": np.round(np.mean(hourly_w_s, axis=0).astype(float),2),
    "cloud cover %": np.round(np.mean(hourly_cl_c, axis=0).astype(float), 2),
    "precipitation (inches)": np.round(np.mean(hourly_precip, axis=0).astype(float),2)
}

hourly_weather_df = pd.DataFrame(data = hourly_data)

### Energy Generation

#### Daily Energy Generation

In [ ]:
d_e_g_rows = []
for item in eia_daily_generation_data:
    d_e_g_rows.append({
        "Date": pd.to_datetime(item["period"]),
        "Energy Type": item["type-name"],
        "Energy Generated (MWh)": item["value"]
    })
eia_daily_generation_df = pd.DataFrame(d_e_g_rows)
eia_daily_generation_df.dropna(inplace=True)

#### Hourly Energy Generation

In [ ]:
h_e_g_rows = []
for item in eia_hourly_generation_data:
    d_e_g_rows.append({
        "Date": pd.to_datetime(item["period"]).date(),
        "Hour": pd.to_datetime(item["period"]).hour,
        "Energy Type": item["type-name"],
        "Energy Generated (MWh)": item["value"]
    })
eia_hourly_generation_df = pd.DataFrame(d_e_g_rows)
eia_hourly_generation_df.dropna(inplace=True)

## Database Creation

### Energy Demand

#### Daily Demand

In [ ]:
eia_daily_df.to_sql("NC_DAILY_ELECTRICITY_DEMAND", conn, if_exists='replace', index=False, method='multi');
%sql sqlite:///North_Carolina_Energy_DB.db

#### Hourly Demand

In [ ]:
eia_hourly_df.to_sql("NC_HOURLY_ELECTRICITY_DEMAND", conn, if_exists='replace', index=False, method='multi');
%sql sqlite:///North_Carolina_Energy_DB.db

### Weather
#### Daily Weather

In [ ]:
daily_weather_df.to_sql("NC_DAILY_WEATHER_DATA", conn, if_exists='replace', index=False, method='multi');

#### Hourly Weather

In [ ]:
hourly_weather_df.to_sql("NC_HOURLY_WEATHER_DATA", conn, if_exists='replace', index=False, method='multi');

### Energy Generation

#### Daily Energy Generation

In [ ]:
eia_daily_generation_df.to_sql("NC_DAILY_ENERGY_GENERATION_DATA", conn, if_exists='replace', index=False, method='multi');

#### Hourly Energy Generation

In [ ]:
eia_hourly_generation_df.to_sql("NC_HOURLY_ENERGY_GENERATION_DATA", conn, if_exists='replace', index=False, method='multi');

## SQL Analysis

In [ ]:
%sql select * from NC_DAILY_ELECTRICITY_DEMAND where "Demand (MWh)" > 800000 limit 10;

In [ ]:
%sql select * from NC_HOURLY_ELECTRICITY_DEMAND order by date  limit 24;

In [ ]:
%sql select * from NC_DAILY_WEATHER_DATA limit 10;

In [ ]:
%sql select * from NC_HOURLY_WEATHER_DATA order by date asc limit 24;

In [ ]:
%sql select * from NC_HOURLY_ENERGY_GENERATION_DATA order by date desc limit 24;